In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

import os
import io
import av
from pathlib import Path

import h5py
import tifffile as tiff
import glob

import subprocess
import tempfile

## Folder Setup


In [2]:
FIJI_APP = "/Users/vuhepola/Desktop/Fiji"
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
FISBe_DIR = DATA_DIR / "FISBe"
FlyLight_DIR = DATA_DIR / "FlyLight"
FANC_DIR = DATA_DIR / "FANC"

print(FIJI_APP)
print(PROJECT_DIR)
print(DATA_DIR)
print(FISBe_DIR)
print(FlyLight_DIR)
print(FANC_DIR)

/Users/vuhepola/Desktop/Fiji
/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly
/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data
/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/FISBe
/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/FlyLight
/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/FANC


## File Tracking


Get all .npz files for a specific line


In [3]:
target_line = "SS04748"
target_line_dir =  FlyLight_DIR /  target_line

npz_files = list(target_line_dir.glob(f"{target_line}*.npz"))

info_arr = []

for f in npz_files:
    file_name = f.name

    info = file_name.split("-")

    line = info[0]
    slide_code = info[1]

    d = {
        "path":f.absolute(),
        "line":line,
        "slide_code":slide_code,
    }

    print(d)
    
    info_arr.append(d)

{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/FlyLight/SS04748/SS04748-20170324_20_I3.npz'), 'line': 'SS04748', 'slide_code': '20170324_20_I3.npz'}
{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/FlyLight/SS04748/SS04748-20170324_20_I2.npz'), 'line': 'SS04748', 'slide_code': '20170324_20_I2.npz'}
{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/FlyLight/SS04748/SS04748-20170324_20_I1.npz'), 'line': 'SS04748', 'slide_code': '20170324_20_I1.npz'}
{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/FlyLight/SS04748/SS04748-20170324_20_I5.npz'), 'line': 'SS04748', 'slide_code': '20170324_20_I5.npz'}
{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/FlyLight/SS04748/SS04748-20170324_20_I4.npz'), 'line': 'SS04748', 'slide_code': '20170324_20_I4.npz'}


## Load and Visualize


Load one file and visualize it with napari


In [4]:
output_npz_path = info_arr[0]['path']


print("Loading data...")
archive = np.load(output_npz_path)
volume = archive['image_data']  # Shape: (127, 4, 1024, 1024)

print(volume.shape)

# Channels 0, 1, 2 are MCFO colors. Channel 3 is the Nc82 background.
signal = volume[:, 0:3, :, :]
reference = volume[:, 3, :, :]

# Normalize
# Convert to float32 and scale to [0, 1], 99.9th percentile
p99 = np.percentile(signal, 99.9)
signal_normalized = np.clip(signal.astype(np.float32) / p99, 0, 1)

print(f"Normalized Signal Shape: {signal_normalized.shape}")
print(f"Min: {signal_normalized.min()}, Max: {signal_normalized.max()}")

Loading data...
(129, 4, 1024, 1024)
Normalized Signal Shape: (129, 3, 1024, 1024)
Min: 0.0, Max: 1.0


In [5]:
import napari

viewer = napari.Viewer(ndisplay=3)

# 3 MCFO channels as a blended RGB-like composite
# Add info of anisotropy using the "scale" parameter, (Z, Y, X)
viewer.add_image(
    signal_normalized[:, 0, :, :], 
    name="MCFO Red", 
    colormap="red", 
    blending="additive",
    scale=(1.0, 0.52, 0.52)  # Corrects the squished Z-axis
)
viewer.add_image(
    signal_normalized[:, 1, :, :], 
    name="MCFO Green", 
    colormap="green", 
    blending="additive",
    scale=(1.0, 0.52, 0.52)
)
viewer.add_image(
    signal_normalized[:, 2, :, :], 
    name="MCFO Blue", 
    colormap="blue", 
    blending="additive",
    scale=(1.0, 0.52, 0.52)
)
viewer.add_image(
    reference, 
    name="Background", 
    colormap="darkgray", 
    blending="additive",
    scale=(1.0, 0.52, 0.52)
)

napari.run()